<a href="https://colab.research.google.com/github/vitevo/ai-qualitative-research-notebooks/blob/main/clasificacion_respuestas_huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación de respuestas abiertas con un modelo de Hugging Face (PyTorch)

**Objetivo.** Clasificar respuestas abiertas de estudiantes sobre el uso de IA en la investigación con un modelo abierto de Hugging Face que corre sobre **PyTorch**, y comparar el resultado con la codificación hecha por una persona o por un LLM vía API.

**Qué hace este cuaderno**
1. Carga un modelo multilingüe de *zero-shot classification* (clasifica con tus categorías sin entrenamiento previo).
2. Clasifica un conjunto pequeño de respuestas.
3. Compara la clasificación del modelo con otras codificaciones (porcentaje de acuerdo y kappa de Cohen).
4. (Opcional) Genera *embeddings* para ver qué respuestas se parecen entre sí.

**Datos.** Las respuestas incluidas son **ficticias**, escritas para practicar. Si usas datos reales, anonimízalos y no los subas a un repositorio público.

**Cómo usarlo.** En Colab, ve a *Entorno de ejecución → Cambiar tipo de entorno de ejecución* y elige **GPU T4** (más rápido, aunque también funciona sin GPU). Luego ejecuta las celdas en orden con ▶.

## 1. Instalar y comprobar las bibliotecas

In [ ]:
!pip -q install transformers sentence-transformers scikit-learn

import torch
import transformers
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU disponible:", torch.cuda.is_available())
device = 0 if torch.cuda.is_available() else -1

## 2. Datos de práctica

Cada fila es una respuesta abierta ficticia. La columna `codigo_humano` simula una codificación hecha por una persona; la columna `codigo_api`, una hecha con un LLM vía API (por ejemplo, GPT o Claude). Reemplázalas por tus propias codificaciones cuando tengas datos.

Para usar tus datos, sube un CSV al panel *Archivos* (a la izquierda) con las columnas `respuesta`, `codigo_humano` y `codigo_api`, y cárgalo con `df = pd.read_csv("tu_archivo.csv")` en lugar de la lista de ejemplo.

In [ ]:
import pandas as pd

datos = [
    ("Ahora me siento capaz de plantear una pregunta de investigación sin ayuda.", "autoeficacia", "autoeficacia"),
    ("Con la IA entendí cómo estructurar el marco teórico y ya lo puedo hacer solo.", "autoeficacia", "autoeficacia"),
    ("Confío más en mi forma de analizar los datos que al inicio del curso.", "autoeficacia", "autoeficacia"),
    ("Sin la IA no sabría por dónde empezar a escribir mi tesis.", "dependencia", "dependencia"),
    ("Siento que copio lo que me dice el chat y no aprendo realmente.", "dependencia", "dependencia"),
    ("Me da miedo entregar algo que no haya revisado antes con la IA.", "dependencia", "autoeficacia"),
    ("Le dedico más horas al trabajo porque quiero que quede bien.", "compromiso", "compromiso"),
    ("Reviso cada fuente que me sugiere la herramienta antes de citarla.", "compromiso", "compromiso"),
    ("Me motiva el tema y busco lecturas por mi cuenta fuera de clase.", "compromiso", "compromiso"),
    ("Propuse a mi grupo hacer las entrevistas en otra comunidad.", "participacion", "participacion"),
    ("Presenté los avances del proyecto en el semillero de investigación.", "participacion", "participacion"),
    ("Tomé la iniciativa de diseñar la encuesta del equipo.", "participacion", "compromiso"),
]
df = pd.DataFrame(datos, columns=["respuesta", "codigo_humano", "codigo_api"])
df

## 3. Definir las categorías

En *zero-shot classification* las categorías se escriben como frases. El modelo estima qué tan probable es que cada respuesta exprese cada frase. La redacción de las etiquetas influye mucho en el resultado: funciona como un codebook en miniatura.

In [ ]:
categorias = {
    "autoeficacia": "confianza del estudiante en su propia capacidad para investigar",
    "dependencia": "dependencia o inseguridad frente a la inteligencia artificial",
    "compromiso": "esfuerzo, dedicación y compromiso con el trabajo académico",
    "participacion": "participación activa e iniciativa en el proceso de investigación",
}
etiquetas = list(categorias.values())
de_etiqueta_a_codigo = {v: k for k, v in categorias.items()}

## 4. Cargar el modelo y clasificar

Se usa un modelo multilingüe (incluye español) entrenado para inferencia de lenguaje natural. La primera ejecución descarga el modelo, lo que puede tardar uno o dos minutos.

In [ ]:
from transformers import pipeline

clasificador = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    device=device,
)

resultados = clasificador(
    df["respuesta"].tolist(),
    candidate_labels=etiquetas,
    hypothesis_template="Este texto expresa {}.",
)

df["codigo_modelo"] = [de_etiqueta_a_codigo[r["labels"][0]] for r in resultados]
df["confianza"] = [round(r["scores"][0], 2) for r in resultados]
df[["respuesta", "codigo_humano", "codigo_modelo", "confianza"]]

## 5. Comparar codificaciones

- **Porcentaje de acuerdo**: proporción de respuestas con el mismo código.
- **Kappa de Cohen**: acuerdo corregido por el azar (0 = azar; 0,41–0,60 moderado; 0,61–0,80 sustancial; más de 0,80 casi perfecto, según Landis y Koch, 1977).
- **Tabla de contingencia**: muestra en qué categorías se confunden.

In [ ]:
from sklearn.metrics import cohen_kappa_score

def comparar(a, b, nombre_a, nombre_b):
    acuerdo = (df[a] == df[b]).mean()
    kappa = cohen_kappa_score(df[a], df[b])
    print(f"{nombre_a} vs. {nombre_b}: acuerdo = {acuerdo:.0%} | kappa = {kappa:.2f}")

comparar("codigo_humano", "codigo_modelo", "Humano", "Modelo HF")
comparar("codigo_humano", "codigo_api", "Humano", "LLM vía API")
comparar("codigo_api", "codigo_modelo", "LLM vía API", "Modelo HF")

pd.crosstab(df["codigo_humano"], df["codigo_modelo"], rownames=["Humano"], colnames=["Modelo HF"])

In [ ]:
# Respuestas en desacuerdo: aquí está lo interesante para el análisis
df[df["codigo_humano"] != df["codigo_modelo"]][["respuesta", "codigo_humano", "codigo_modelo", "confianza"]]

## 6. (Opcional) Embeddings: ¿qué respuestas se parecen?

Un *embedding* convierte cada respuesta en un vector de números, es decir, un tensor. Respuestas con significado parecido quedan cerca. Sirve para explorar agrupaciones antes de codificar o para revisar la coherencia de un tema.

In [ ]:
from sentence_transformers import SentenceTransformer, util

modelo_emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = modelo_emb.encode(df["respuesta"].tolist(), convert_to_tensor=True)
print("Forma del tensor de embeddings:", tuple(embeddings.shape))  # (respuestas, dimensiones)

similitud = util.cos_sim(embeddings, embeddings)
for i, texto in enumerate(df["respuesta"]):
    valores = similitud[i].clone()
    valores[i] = -1  # ignorar la propia respuesta
    j = int(torch.argmax(valores))
    print(f"[{i}] {texto[:60]}...")
    print(f"     más parecida -> [{j}] {df['respuesta'][j][:60]}... ({valores[j]:.2f})\n")

## 7. Guardar resultados

In [ ]:
df.to_csv("clasificacion_resultados.csv", index=False)
print("Archivo guardado: clasificacion_resultados.csv (panel Archivos, a la izquierda)")

## Notas metodológicas

- El modelo **no reemplaza** la codificación interpretativa: es un punto de comparación. Los desacuerdos son material analítico, no errores que haya que ocultar.
- Con datos reales, informa el modelo, la versión, la redacción exacta de las etiquetas y la plantilla de hipótesis, para que el procedimiento sea reproducible.
- Un modelo abierto como este procesa los textos dentro de Colab; un LLM vía API los envía a los servidores del proveedor. Considera esa diferencia en el consentimiento informado y en la revisión ética.